<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/50_build_app_artefacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# =============================================================================
# 50_build_app_artefacts.ipynb
# Purpose: consolidate baseline + keyword strategy outputs into artefacts
# Location: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
# These artefacts are what the Streamlit demo app consumes.
# =============================================================================

# --- Setup: mount Drive and imports -------------------------------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
import math
import re

# --- Project paths ------------------------------------------------------------
PROJECT_DIR   = Path("/content/drive/MyDrive/gt-markets")
SRC_EXTRACT   = PROJECT_DIR / "app-demo" / "extracted"      # daily/weekly model outputs
DST_ARTE      = PROJECT_DIR / "AppDemo"  / "artefacts"      # artefacts for the app
DATA_PROCESSED= PROJECT_DIR / "data"      / "processed"     # merged financial+trend data

DST_ARTE.mkdir(parents=True, exist_ok=True)

# --- Configuration ------------------------------------------------------------
ASSETS   = ["GOLD", "BTC", "OIL", "USDCNY"]
FREQ_MAP = {"daily": "D", "weekly": "W"}
ANNUAL   = {"D": 252, "W": 52}
COST_BPS = 5   # cost per turnover (bps)

# Canonical asset mapping
ASSET_CANON = {"gold":"GOLD","btc":"BTC","oil":"OIL","usdcny":"USDCNY"}

# =============================================================================
# Baseline strategies (SMA, EMA, etc.)
# These baselines are quick technical indicator strategies applied directly
# to the price series. They are rebuilt fresh for each asset/frequency.
# =============================================================================

def load_price_data():
    """Load merged processed dataset for prices."""
    files = sorted(DATA_PROCESSED.glob("merged_financial_trends_data_*.csv"))
    if not files:
        raise FileNotFoundError("No merged data files in processed/")
    df = pd.read_csv(files[-1], parse_dates=["Date"])
    df = df.set_index("Date").sort_index()
    return df

def baseline_strategies(close: pd.Series, freq: str):
    """Generate simple SMA/EMA crossover baselines."""
    out = {}
    # SMA 30 vs SMA 100
    sma30, sma100 = close.rolling(30).mean(), close.rolling(100).mean()
    pos_sma = (sma30 > sma100).astype(int)
    out["BASE_SMA"] = pos_sma
    # EMA 12 vs EMA 26
    ema12, ema26 = close.ewm(span=12).mean(), close.ewm(span=26).mean()
    pos_ema = (ema12 > ema26).astype(int)
    out["BASE_EMA"] = pos_ema
    return out

def equity_and_metrics(close: pd.Series, pos: pd.Series, freq: str):
    """Compute equity + KPIs given close price and binary position."""
    ret  = close.pct_change(fill_method=None).fillna(0.0)
    pos  = pos.fillna(0.0).clip(0,1)
    turns= pos.diff().abs().fillna(0.0)
    cost = turns * (COST_BPS/1e4)
    strat= (pos.shift(1).fillna(0.0) * ret) - cost
    eq   = (1.0 + strat).cumprod()
    ann  = ANNUAL[freq]
    mu   = strat.mean() * ann
    sd   = strat.std(ddof=0) * math.sqrt(ann)
    sharpe = mu/sd if sd>0 else np.nan
    maxdd  = (eq/eq.cummax()-1).min()
    return eq, {"Return_Ann": mu, "Vol_Ann": sd, "Sharpe": sharpe, "MaxDD": maxdd}

def build_baseline_artefacts():
    df = load_price_data()
    price_cols = {
        "GOLD": "GC=F Close",
        "BTC": "BTC-USD Close",
        "OIL": "CL=F Close",
        "USDCNY": "USDCNY=X Close"
    }
    for asset, col in price_cols.items():
        close = df[col].dropna()
        for freq in ["D","W"]:
            pos_dict = baseline_strategies(close, freq)
            rows=[]
            for strat, pos in pos_dict.items():
                eq,mets=equity_and_metrics(close,pos,freq)
                rows.append({"asset":asset,"freq":freq,"strategy":strat,**mets})
                out_dir=DST_ARTE/asset
                (out_dir/"figs").mkdir(parents=True,exist_ok=True)
                sig=pos.diff().fillna(0).replace({1:1,-1:0}).astype(int)
                pd.DataFrame({
                    "Date":close.index,"signal":sig,"position":pos,
                    "Close":close.values,"strategy":strat
                }).to_csv(out_dir/f"signals_{strat}_{freq}.csv",index=False)
                ax=eq.plot(figsize=(6,3),title=f"{asset} – {strat} – {freq}")
                ax.grid(True,alpha=0.3)
                ax.get_figure().savefig(out_dir/"figs"/f"{strat}_{freq}.png",dpi=140)
                plt.close(ax.get_figure())
            pd.DataFrame(rows).to_csv(DST_ARTE/asset/f"metrics_baseline_{freq}.csv",index=False)
    print("✅ Baseline artefacts built.")

# =============================================================================
# Keyword model artefacts (from extracted runs)
# These are outputs from ML models trained with keyword features.
# =============================================================================

def canon_asset(s: str) -> str:
    s = s.lower()
    for k,v in ASSET_CANON.items():
        if k in s or f"{k}=" in s:
            return v
    return None

def safe_read_csv(p: Path) -> pd.DataFrame:
    try:
        df = pd.read_csv(p)
    except Exception:
        return pd.DataFrame()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
    return df

def infer_position(df: pd.DataFrame) -> pd.Series:
    cols = {c.lower():c for c in df.columns}
    if "position" in cols:
        pos = pd.to_numeric(df[cols["position"]], errors="coerce").fillna(0.0)
        return (pos>0.5).astype(float)
    elif "prob_up" in cols:
        return (pd.to_numeric(df[cols["prob_up"]], errors="coerce")>0.5).astype(float)
    else:
        return pd.Series(dtype=float)

def strategy_name_from_path(p: Path) -> str:
    kw = re.sub(r"[^A-Za-z0-9]+","_",p.stem).upper()
    return ("KW_"+kw)[:32]

def build_keyword_artefacts():
    rows_by_key={}
    for subdir,freq in FREQ_MAP.items():
        src=SRC_EXTRACT/subdir
        if not src.exists(): continue
        for csv in src.rglob("*.csv"):
            df=safe_read_csv(csv)
            if df.empty: continue
            asset=canon_asset(csv.name) or canon_asset(str(csv))
            if not asset: continue
            pos=infer_position(df)
            if pos.empty: continue
            close_col=None
            for c in df.columns:
                if c.lower() in ("close","price","adjclose","adj_close"):
                    close_col=c; break
            if not close_col: continue
            eq,mets=equity_and_metrics(df[close_col],pos,freq)
            strat=strategy_name_from_path(csv)
            out_dir=DST_ARTE/asset
            (out_dir/"figs").mkdir(parents=True,exist_ok=True)
            sig=pos.diff().fillna(0.0).replace({1:1,-1:0}).astype(int)
            pd.DataFrame({
                "Date":df["Date"],"signal":sig,"position":pos.astype(int),
                "Close":pd.to_numeric(df[close_col],errors="coerce"),
                "strategy":strat
            }).to_csv(out_dir/f"signals_{strat}_{freq}.csv",index=False)
            ax=eq.plot(figsize=(6,3),title=f"{asset} – {strat} – {freq}")
            ax.grid(True,alpha=0.25)
            ax.get_figure().savefig(out_dir/"figs"/f"{strat}_{freq}.png",dpi=140)
            plt.close(ax.get_figure())
            rows_by_key[(asset,freq,strat)]={"asset":asset,"freq":freq,"strategy":strat,**mets}
    if not rows_by_key:
        print("⚠️ No keyword runs found.")
        return
    metrics=pd.DataFrame(rows_by_key.values())
    for asset in metrics["asset"].unique():
        for freq in ["D","W"]:
            out_dir=DST_ARTE/asset
            part=metrics.query("asset==@asset and freq==@freq").copy()
            if part.empty: continue
            part.to_csv(out_dir/f"metrics_keywords_{freq}.csv",index=False)
    print("✅ Keyword artefacts built.")

# =============================================================================
# Cleanup and validation
# =============================================================================

def cleanup_folders():
    """Ensure only uppercase asset dirs remain (merge Gold→GOLD, Oil→OIL)."""
    for child in DST_ARTE.iterdir():
        if child.is_dir():
            canon = ASSET_CANON.get(child.name.lower())
            if canon and child.name != canon:
                target=DST_ARTE/canon
                target.mkdir(parents=True,exist_ok=True)
                for f in child.rglob("*"):
                    rel=f.relative_to(child)
                    (target/rel.parent).mkdir(parents=True,exist_ok=True)
                    if f.is_file():
                        shutil.move(str(f), str(target/rel))
                shutil.rmtree(child, ignore_errors=True)
    print("✅ Folders cleaned.")

def validate():
    rows=[]
    for asset in ASSETS:
        a_dir=DST_ARTE/asset
        for freq in ["D","W"]:
            rows.append({
                "asset":asset,"freq":freq,
                "baseline":"✓" if (a_dir/f"metrics_baseline_{freq}.csv").exists() else "✗",
                "keywords":"✓" if (a_dir/f"metrics_keywords_{freq}.csv").exists() else "✗",
                "signals":"✓" if any(a_dir.glob(f"signals_*_{freq}.csv")) else "✗",
                "figs":"✓" if any((a_dir/"figs").glob(f"*_{freq}.png")) else "✗"
            })
    print(pd.DataFrame(rows).to_string(index=False))

# =============================================================================
# Run everything
# =============================================================================

build_baseline_artefacts()
build_keyword_artefacts()
cleanup_folders()
validate()
print("✅ All artefacts ready at:", DST_ARTE)


Mounted at /content/drive
✅ Baseline artefacts built.
⚠️ No keyword runs found.
✅ Folders cleaned.
 asset freq baseline keywords signals figs
  GOLD    D        ✓        ✓       ✓    ✓
  GOLD    W        ✓        ✓       ✓    ✓
   BTC    D        ✓        ✓       ✓    ✓
   BTC    W        ✓        ✓       ✓    ✓
   OIL    D        ✓        ✓       ✓    ✓
   OIL    W        ✓        ✓       ✓    ✓
USDCNY    D        ✓        ✓       ✓    ✓
USDCNY    W        ✓        ✓       ✓    ✓
✅ All artefacts ready at: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
